In [1]:
import pypdf

def extract_text(path):
    reader = pypdf.PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

print("Ready to extract text from PDF files.")

Ready to extract text from PDF files.


In [2]:
from pathlib import Path

pdf_path = Path("../data/raw/ML_paper.pdf")

text = extract_text(str(pdf_path))

print("Text length:", len(text))
print(text[:500])

Text length: 32631
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent o


In [3]:
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        
        start += chunk_size - overlap
        
    return chunks

In [4]:
chunks = chunk_text(text)

print("Number of chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0])

Number of chunks: 47

First chunk preview:

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and c


In [5]:
import json
from pathlib import Path

output_path = Path("../data/processed/chunks.jsonl")

with open(output_path, "w") as f:
    for i, chunk in enumerate(chunks):
        record = {
            "chunk_id": i,
            "text": chunk
        }
        f.write(json.dumps(record) + "\n")

print("Chunks saved to:", output_path)

Chunks saved to: ../data/processed/chunks.jsonl


In [6]:
pip install openai python-dotenv

You should consider upgrading via the '/Users/ossamamorjani/Documents/dev/llm-pdf-classifier/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", api_key[:8], "...")

API key loaded: sk-proj- ...


Testing embeddings 

In [8]:
from openai import OpenAI

client = OpenAI()

chunk_example = chunks[0]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunk_example
)

embedding_vector = response.data[0].embedding

print("Vector length:", len(embedding_vector))
print(embedding_vector[:10])

Vector length: 1536
[0.0233306884765625, 0.005016326904296875, -0.0211639404296875, 0.0226287841796875, 0.014068603515625, -0.038360595703125, 0.01081085205078125, 0.05633544921875, -0.054046630859375, 0.003849029541015625]


Vector indexing

In [9]:
pip install faiss-cpu

You should consider upgrading via the '/Users/ossamamorjani/Documents/dev/llm-pdf-classifier/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


Need to generate embeddings for all the chunks

In [10]:
import numpy as np
embeddings = []

for chunk in chunks:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk
    )
    
    vector = response.data[0].embedding
    embeddings.append(vector)

print("Total embeddings:", len(embeddings))
print("Vector dimension:", len(embeddings[0]))

Total embeddings: 47
Vector dimension: 1536


In [11]:
embedding_matrix = np.array(embeddings).astype("float32")
print("Embedding matrix shape:", embedding_matrix.shape)

Embedding matrix shape: (47, 1536)


Now onto building the search index

In [12]:
import faiss

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("Vectors in index:", index.ntotal)

Vectors in index: 47


Now I'll simulate a user asking something. 

In [13]:
query = "What is the transformer architecture?"

query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_vector = np.array([query_embedding.data[0].embedding]).astype("float32")

k = 3
distances, indices = index.search(query_vector, k)

print("Top chunks:", indices)

Top chunks: [[10  9  6]]


Need to retrieve the actual text

In [14]:
for i in indices[0]:
    print(f"\nChunk {i}:\n")
    print(chunks[i])


Chunk 10:

 of a stack of N = 6 identical layers. Each layer has two
sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-
2Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x+ Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder: The decoder is also composed of a stack of N = 6identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a

Chunk 9:

e an encoder-decoder structure [5, 2, 29].
Here, the encoder maps an input sequence of symbol representations (x1,...,x n) to a sequence
of continuous representations z = (z1,.

In [15]:

all_texts = chunks

all_embeddings = []
for text in all_texts:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    all_embeddings.append(response.data[0].embedding)

embedding_matrix = np.array(all_embeddings, dtype="float32")

print("Shape:", embedding_matrix.shape)

Shape: (47, 1536)


So far, I have built something that can retrieve information, as in relevant chunks. Now, I want to progress into building a real AI Assistant - the retrieval augmented generation RAG.

In [16]:
retrieved_chunks = [chunks[i] for i in indices[0]]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Retrieved chunk {i+1} ---\n")
    print(chunk[:500])


--- Retrieved chunk 1 ---

 of a stack of N = 6 identical layers. Each layer has two
sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-
2Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x+ Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. 

--- Retrieved chunk 2 ---

e an encoder-decoder structure [5, 2, 29].
Here, the encoder maps an input sequence of symbol representations (x1,...,x n) to a sequence
of continuous representations z = (z1,...,z n). Given z, the decoder then generates an output
sequence (y1,...,y m) of symbols one element at a time. At each step the model is auto-regressive
[9], consuming the previously generated symbols as additional input when generating the next.
The Transformer foll

Building context for the LLM

In [17]:
context = "\n\n".join(retrieved_chunks)

print(context[:1000])

 of a stack of N = 6 identical layers. Each layer has two
sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-
2Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x+ Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder: The decoder is also composed of a stack of N = 6identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a

e an encoder-decoder structure [5, 2, 29].
Here, the encoder maps an input sequence of symbol representations (x1,...,x n) to a sequence
of continuous representations z = (z1,...,z n). Given z, the 

In [18]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You answer questions using the provided context only."
        },
        {
            "role": "user",
            "content": f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}
"""
        }
    ]
)

print(response.choices[0].message.content)

The transformer architecture consists of an encoder-decoder structure with stacked layers. The encoder is composed of a stack of N = 6 identical layers, each with two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network. It employs residual connections around each sub-layer followed by layer normalization. The outputs of all sub-layers, as well as the embedding layers, have a dimension of dmodel = 512.

The decoder is also made up of a stack of N = 6 identical layers and includes the same two sub-layers as the encoder. Additionally, the decoder incorporates mechanisms for generating an output sequence of symbols one element at a time in an auto-regressive manner, utilizing previously generated symbols as inputs for the next generation step. 

Overall, the transformer architecture enables enhanced parallelization and efficiency, leading to high-quality translation capabilities after relatively short training times on powerful hardwar